# Module 2 - NDVI Crop Stress Classification

Classifies farm-field imagery into **healthy / moderate_stress / severe_stress** using NDVI statistics computed from paired RGB and NIR images (Agriculture-Vision 2017 mini-scale release).

**Pipeline**

1. Download and extract the dataset from the public `intelinair-data-releases` S3 bucket
2. Compute NDVI per image and reduce each map to 6 summary features
3. Experiment A - Random Forest with balanced class weights
4. Experiment B - Random Forest on SMOTE-resampled data (tested and rejected)
5. Final model - Experiment A's configuration, verified and saved to `module2_rf.pkl`

**Features** `ndvi_mean`, `ndvi_std`, `ndvi_min`, `ndvi_max`, `ndvi_p25`, `ndvi_p75` | **Target** `label`

**Split** `train_test_split(test_size=0.2, stratify=y, random_state=42)` - identical in every experiment, so the three models are directly comparable.

## 1. Data acquisition

Downloads `data2017_miniscale.tar.gz` from the public Agriculture-Vision S3 bucket using an unsigned boto3 client, so no AWS credentials are needed, then extracts it into `/kaggle/working/`.

This yields roughly 8,300 paired RGB and NIR images under `field_images/`, plus per-condition binary masks under `field_labels/` - one subfolder per condition: drydown, nutrient_deficiency, storm_damage, double_plant, planter_skip, endrow, weed_cluster, water, waterway.

In [ ]:
!pip install -q boto3
import boto3, tarfile, os
from botocore import UNSIGNED
from botocore.client import Config

s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))  # public bucket, no credentials needed

bucket = "intelinair-data-releases"
key = "agriculture-vision/cvpr_paper_2020/Dataset/data2017_miniscale.tar.gz"
local_tar = "/kaggle/working/data2017_miniscale.tar.gz"

s3.download_file(bucket, key, local_tar)
print("Download complete")

with tarfile.open(local_tar) as tar:
    tar.extractall("/kaggle/working/")
print("Extraction complete")

base = "/kaggle/working/data2017_miniscale"
rgb_dir = os.path.join(base, "field_images", "rgb")
nir_dir = os.path.join(base, "field_images", "nir")
labels_dir = os.path.join(base, "field_labels")

print("Total RGB images available:", len(os.listdir(rgb_dir)))
print("Total NIR images available:", len(os.listdir(nir_dir)))
print("Label subfolders:", os.listdir(labels_dir))

## 2. NDVI feature extraction

Draws a seeded random sample of 2,000 images and, for each one, computes `NDVI = (NIR - Red) / (NIR + Red + 1e-8)`, then reduces the NDVI map to six summary statistics: mean, standard deviation, min, max, 25th percentile and 75th percentile.

**Labelling rule**, driven by the share of pixels each condition mask covers:

- water or waterway above 15% - image is dropped, not crop-health relevant
- drydown, nutrient_deficiency or storm_damage above 5% - `severe_stress`
- double_plant, planter_skip, endrow or weed_cluster above 2% - `moderate_stress`
- otherwise - `healthy`

The result is `final_df`, also written to `/kaggle/working/final_df_scaled.csv` so the features survive a kernel restart.

**Reproducibility note** the sample is seeded, but `os.listdir` does not guarantee ordering, so a fresh kernel can draw a slightly different 2,000 images. Expect small shifts in row counts and metrics between runs.

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
import os, random
from tqdm import tqdm

random.seed(42)
np.random.seed(42)

SAMPLE_SIZE = 2000
all_filenames = os.listdir(rgb_dir)
sampled_filenames = random.sample(all_filenames, SAMPLE_SIZE)  # true seeded random sample

severe_threshold, moderate_threshold, exclude_threshold = 5.0, 2.0, 15.0
severe_labels = {"drydown", "nutrient_deficiency", "storm_damage"}
moderate_labels = {"double_plant", "planter_skip", "endrow", "weed_cluster"}
exclude_labels = {"water", "waterway"}
label_subfolders = os.listdir(labels_dir)  # read once, not per image

records, skipped = [], 0

for fname in tqdm(sampled_filenames):
    nir_path = os.path.join(nir_dir, fname)
    rgb_path = os.path.join(rgb_dir, fname)
    if not os.path.exists(nir_path):
        skipped += 1
        continue
    try:
        rgb = np.array(Image.open(rgb_path))
        nir = np.array(Image.open(nir_path).convert("L")).astype(np.float32)
        red = rgb[:, :, 0].astype(np.float32)
        ndvi = (nir - red) / (nir + red + 1e-8)
    except Exception:
        skipped += 1
        continue

    mask_name = os.path.splitext(fname)[0] + ".png"
    total_px = ndvi.size
    label_pct = {}
    for lbl in label_subfolders:
        mpath = os.path.join(labels_dir, lbl, mask_name)
        if os.path.exists(mpath):
            mask = np.array(Image.open(mpath))
            label_pct[lbl] = (mask > 0).sum() / total_px * 100

    if any(label_pct.get(l, 0) > exclude_threshold for l in exclude_labels):
        continue  # mostly water/waterway — not crop-health relevant

    if any(label_pct.get(l, 0) > severe_threshold for l in severe_labels):
        stress_label = "severe_stress"
    elif any(label_pct.get(l, 0) > moderate_threshold for l in moderate_labels):
        stress_label = "moderate_stress"
    else:
        stress_label = "healthy"

    records.append({
        "filename": fname,
        "ndvi_mean": ndvi.mean(), "ndvi_std": ndvi.std(),
        "ndvi_min": ndvi.min(), "ndvi_max": ndvi.max(),
        "ndvi_p25": np.percentile(ndvi, 25), "ndvi_p75": np.percentile(ndvi, 75),
        "label": stress_label,
    })

final_df = pd.DataFrame(records)
print(f"Skipped {skipped} images (missing pairing or unreadable)")
print(final_df.shape)
print(final_df["label"].value_counts())

final_df.to_csv("/kaggle/working/final_df_scaled.csv", index=False)  # save now, before anything can expire

## 3. Experiment A - Random Forest with balanced class weights

First model. 200 trees with `class_weight="balanced"`, which reweights each class by inverse frequency - worth doing because the labels land at roughly 46% severe / 37% moderate / 17% healthy.

Trained on the original, un-resampled training split. The test set is never touched by any resampling, in this cell or the next.

Also prints feature importances and dumps the fitted estimator to `ndvi_random_forest.pkl`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import joblib

feature_cols = ["ndvi_mean", "ndvi_std", "ndvi_min", "ndvi_max", "ndvi_p25", "ndvi_p75"]
X = final_df[feature_cols]
y = final_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",   # compensates for the 46/36/18 imbalance confirmed above
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))

cm = confusion_matrix(y_test, y_pred, labels=["healthy", "moderate_stress", "severe_stress"])
print("Confusion matrix (rows=true, cols=predicted, order: healthy/moderate/severe)")
print(cm)

# Which NDVI statistic actually drives the model's decisions — useful for the report
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nFeature importances:")
print(importances)

joblib.dump(rf, "/kaggle/working/ndvi_random_forest.pkl")
final_df.to_csv("/kaggle/working/final_df_scaled.csv", index=False)  # already saved, just confirming both artifacts persist


## 4. Experiment B - SMOTE oversampling (tested, rejected)

Rebuilds the same split, then applies SMOTE to the **training set only** - the test set stays real and untouched, so the comparison against Experiment A is fair. `class_weight` is dropped here because SMOTE has already balanced the classes and stacking both would over-correct.

**Why this is not the final model.** SMOTE lifts raw `healthy` recall, but it sharply increases the error that matters most here: a field that is truly in `severe_stress` being predicted `healthy`. In the run recorded in these outputs that cell goes from 9 cases under Experiment A to 23 under SMOTE. A missed severe-stress field is the expensive failure for a crop-health alert, so Experiment A is kept and this one is not used.

In [ ]:
!pip install -q imbalanced-learn
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

feature_cols = ["ndvi_mean", "ndvi_std", "ndvi_min", "ndvi_max", "ndvi_p25", "ndvi_p75"]
X = final_df[feature_cols]
y = final_df["label"]

# Same split as before (same random_state) so results are directly comparable
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# SMOTE touches ONLY the training set — the test set stays real, untouched data
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", y_train.value_counts().to_dict())
print("After SMOTE:", y_train_res.value_counts().to_dict())

rf_smote = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
    # class_weight dropped — SMOTE already balanced the training set directly,
    # stacking both would over-correct
)
rf_smote.fit(X_train_res, y_train_res)

y_pred = rf_smote.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))

cm = confusion_matrix(y_test, y_pred, labels=["healthy", "moderate_stress", "severe_stress"])
print("Confusion matrix (rows=true, cols=predicted, order: healthy/moderate/severe)")
print(cm)

## 5. Final model

Experiment A's configuration, trained explicitly on the original non-SMOTE split, then saved and verified in the same cell.

- **Split safety** reuses the live `X_train / X_test / y_train / y_test` only after checking that their row indices match a freshly recomputed `train_test_split(test_size=0.2, stratify=y, random_state=42)`. If they do not match, or the kernel has been restarted, it recreates the split from `final_df`, falling back to `final_df_scaled.csv`.
- **Guard against SMOTE leaking in** prints the training class counts, which must show the original imbalance, and prints the `class_weight` actually in use.
- **Model** `RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1)`.
- **Reported** classification report at three decimals, confusion matrix with an explicit label order, and the true `severe_stress` predicted `healthy` cell called out on its own line.
- **Saved** to `/kaggle/working/module2_rf.pkl`, then reloaded with `joblib.load` and re-scored on the same test set to prove the file on disk reproduces identical predictions, report and confusion matrix.


In [ ]:
import os, joblib, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
FEATURE_COLS = ["ndvi_mean", "ndvi_std", "ndvi_min", "ndvi_max", "ndvi_p25", "ndvi_p75"]
LABELS = ["healthy", "moderate_stress", "severe_stress"]
G = globals()
src = G["final_df"] if "final_df" in G else pd.read_csv("/kaggle/working/final_df_scaled.csv")
Xr = src[FEATURE_COLS]
yr = src["label"]
A, B, C, D = train_test_split(Xr, yr, test_size=0.2, stratify=yr, random_state=42)
idx = lambda o: list(o.index) if hasattr(o, "index") else None
have = all(k in G for k in ("X_train", "X_test", "y_train", "y_test"))
match = have and idx(G.get("X_train")) == idx(A) and idx(G.get("X_test")) == idx(B) and idx(G.get("y_train")) == idx(C) and idx(G.get("y_test")) == idx(D)
Xtr, Xte, ytr, yte = (G["X_train"], G["X_test"], G["y_train"], G["y_test"]) if match else (A, B, C, D)
print("Split source:", "REUSED existing X_train/X_test/y_train/y_test from the kernel" if match else "RECREATED with test_size=0.2, stratify=y, random_state=42")
print("Train size:", len(Xtr), "Test size:", len(Xte))
print("Train class counts, must be the ORIGINAL imbalance and NOT SMOTE:", ytr.value_counts().to_dict())
model = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1)
model.fit(Xtr, ytr)
print("class_weight in use:", model.get_params()["class_weight"])
y_pred = model.predict(Xte)
rep = classification_report(yte, y_pred, digits=3)
print(rep)
cm = confusion_matrix(yte, y_pred, labels=LABELS)
print("Confusion matrix rows=true cols=predicted order healthy/moderate_stress/severe_stress")
print(cm)
print("TRUE severe_stress PREDICTED healthy =", cm[2][0], " -- baseline was 6, SMOTE was 20")
path = "/kaggle/working/module2_rf.pkl"
joblib.dump(model, path)
size = os.path.getsize(path)
print("Saved to:", path)
print("File size:", size, "bytes =", round(size/1048576, 2), "MB")
reloaded = joblib.load(path)
y_pred2 = reloaded.predict(Xte)
rep2 = classification_report(yte, y_pred2, digits=3)
print(rep2)
print("Reloaded predictions identical:", bool((y_pred2 == y_pred).all()))
print("Reloaded classification_report identical:", rep == rep2)
print("Reloaded confusion matrix identical:", bool((confusion_matrix(yte, y_pred2, labels=LABELS) == cm).all()))